In [0]:
customers = spark.read.table("02_silver_catalog.default.customers")
exchange_rates= spark.read.table("02_silver_catalog.default.exchange_rates")
invoice_line_items = spark.read.table("02_silver_catalog.default.invoice_line_items")
invoices = spark.read.table("02_silver_catalog.default.invoices")
products = spark.read.table("02_silver_catalog.default.products")
payments = spark.read.table("02_silver_catalog.default.payments")
regions = spark.read.table("02_silver_catalog.default.regions")
display(customers)
display(exchange_rates)
display(invoice_line_items)
display(invoices)
display(products)
display(payments)
display(regions)

In [0]:
tables = [
    "customers",
    "exchange_rates",
    "invoice_line_items",
    "invoices",
    "products",
    "payments"
]

for table in tables:
    df = spark.read.table(f"02_silver_catalog.default.{table}")
    df_clean = df.dropDuplicates()
    df_clean.write.mode("overwrite").saveAsTable(f"02_silver_catalog.default.{table}")

In [0]:
tables = [
    "customers",
    "exchange_rates",
    "invoice_line_items",
    "invoices",
    "products",
    "payments"
]

for table in tables:
    df = spark.read.table(f"02_silver_catalog.default.{table}")
    for col_name, dtype in df.dtypes:
        if dtype.startswith("string"):
            df = df.fillna({col_name: "unknown"})
        elif dtype in ["int", "bigint", "double", "float", "decimal"]:
            df = df.fillna({col_name: 0})
    df_clean = df.dropDuplicates()
    df_clean.write.mode("overwrite").saveAsTable(f"02_silver_catalog.default.{table}")

In [0]:
from pyspark.sql.functions import col, to_date

exchange_rates = spark.read.table("02_silver_catalog.default.exchange_rates") \
    .withColumn("date", to_date(col("date"), "yyyy/M/d"))
exchange_rates.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("02_silver_catalog.default.exchange_rates")

invoices = spark.read.table("02_silver_catalog.default.invoices") \
    .withColumn("invoice_date", to_date(col("invoice_date"), "yyyy/M/d"))
invoices.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("02_silver_catalog.default.invoices")

payments = spark.read.table("02_silver_catalog.default.payments") \
    .withColumn("payment_date", to_date(col("payment_date"), "yyyy/M/d"))
payments.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("02_silver_catalog.default.payments")